In [ ]:
%pip install -U langchain langchain-core langchain-community langchain-experimental langchain-text-splitters langchain-neo4j langchain-ollama neo4j python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 9.8 MB/s  0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3
  Attempting uninstall: langchain-neo4j━━━━━━━━━ 0/3 [langchain-core]
    Found existing installation: langchain-neo4j 0.9.00/3 [langchain-core]
    Uninstalling langchain-neo4j-0.9.0:━━━━━ 0/3 [langchain-core]
      Successfully uninstalled langchain-neo4j-0.9.02m0/3 [langchain-core]
  Attempting uninstall: langchain━━━━━━━━━━━ 0/3 [langchain-core]
    Found existing installation: langchain 1.3.4 0/3 [langchain-core]
    Uninstalling langchain-1.3.4:━━━━━━━━━━━ 0/3 [langchain-core]
      Successfully uninstalled langchain-1.3.40m 0/3 [langchain-core]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langchain]]
Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
import os

from pydantic import BaseModel, Field
from neo4j import GraphDatabase, Driver

from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_neo4j import Neo4jGraph, Neo4jVector
from langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars

from langchain_experimental.graph_transformers import LLMGraphTransformer
from yfiles_jupyter_graphs import GraphWidget


/tmp/ipykernel_316162/3650462810.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
/tmp/ipykernel_316162/3650462810.py:18: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


In [ ]:
load_dotenv()

True

In [8]:
graph = Neo4jGraph(database="cloud")

In [7]:
loader = TextLoader(file_path="../data/chapters/chapter_1.txt")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=200)
documents = text_splitter.split_documents(documents=docs)

In [9]:
print(documents)

[Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='I. A SCANDAL IN BOHEMIA\n\n\nI.'), Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='To Sherlock Holmes she is always _the_ woman. I have seldom heard him\nmention her under any other name. In his eyes she eclipses and\npredominates the whole of her sex. It was not that he felt any emotion\nakin to love for Irene Adler. All emotions, and that one particularly,\nwere abhorrent to his cold, precise but admirably balanced mind. He\nwas, I take it, the most perfect reasoning and observing machine that\nthe world has seen, but as a lover he would have placed himself in a\nfalse position. He never spoke of the softer passions, save with a gibe\nand a sneer. They were admirable things for the observer—excellent for\ndrawing the veil from men’s motives and actions. But for the trained'), Document(metadata={'source': '../data/chapters/chapter_1.txt'}, page_content='and a sneer. They were a

In [1]:
import requests
print(requests.get("http://192.168.178.67:11434/api/tags", timeout=5).json())

ConnectionError: HTTPConnectionPool(host='192.168.178.67', port=11434): Max retries exceeded with url: /api/tags (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7f569dc75010>: Failed to establish a new connection: [Errno 113] No route to host'))

In [63]:
# llm = ChatOllama(model="llama3.1:8b", base_url="http://localhost:11434", temperature=0, reasoning=False)

llm = ChatOllama(model="mistral:latest", temperature=0, reasoning=False, keep_alive="24h")


# llm = ChatOllama(model="", temperature=0,  reasoning=False)192.168.178.51

llm.invoke("Say hello in one sentence.")

AIMessage(content=' Hello there! How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'mistral:latest', 'created_at': '2026-06-12T17:00:50.141308507Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6880879216, 'load_duration': 5166036952, 'prompt_eval_count': 11, 'prompt_eval_duration': 451649119, 'eval_count': 11, 'eval_duration': 1257204888, 'logprobs': None, 'model_name': 'mistral:latest', 'model_provider': 'ollama'}, id='lc_run--019ebcc7-5f7b-7811-8750-3edd560beb59-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 11, 'output_tokens': 11, 'total_tokens': 22})

In [9]:
from langchain_experimental.graph_transformers.llm import SystemMessage, HumanMessagePromptTemplate, PromptTemplate, ChatPromptTemplate

In [10]:
system_prompt = """ You are a law and crime AI assistant with expertise in natural language processing, 
specifically in named entity recognition (NER) for cases, persons, relationships, places and events. 
You are building a knowledge graph database. 

Your task is to identify and extract all 
case-related entities and relationships from the given text.

Return nothing but the nodes and relations in a JSON format.
"""

# You must generate the output in a JSON format containing a list with JSON objects.

system_message = SystemMessage(content=system_prompt)

user_prompt = """
# Task: Analyse the following text and extract all relevant entities and relationships. 



# output: JSON Objects following this schema: 
examples: {
  "nodes": [
    {
      "id": "1",
      "type": "Person",
      "properties": {
        "description": "Sherlock Holmes",
        "context": "A precise, cold and balanced reasoning machine, deeply attracted to the study of crime",
        "traits": ["logical", "emotionally detached", "bohemian", "observant"],
        "physical_description": "tall, spare figure",
        "habits": ["cocaine use", "pacing", "studying crime"]
      }
    },
    {
      "id": "2",
      "type": "Person",
      "properties": {
        "description": "Irene Adler",
        "context": "The only woman of significance to Holmes, described as late and of dubious memory",
        "status": "deceased"
      }
    },
    {
      "id": "3",
      "type": "Person",
      "properties": {
        "description": "Dr. Watson",
        "context": "Narrator, former companion of Holmes, medical doctor returned to civil practice",
        "status": "married",
        "occupation": "medical doctor"
      }
    },
    {
      "id": "4",
      "type": "Person",
      "properties": {
        "description": "Trepoff",
        "context": "Victim of a murder case in Odessa"
      }
    },
    {
      "id": "5",
      "type": "Person",
      "properties": {
        "description": "Atkinson Brothers",
        "context": "Involved in a singular tragedy at Trincomalee"
      }
    },
    {
      "id": "6",
      "type": "Organization",
      "properties": {
        "description": "Reigning Family of Holland",
        "context": "Royal family of Holland for whom Holmes accomplished a delicate mission"
      }
    },
    {
      "id": "7",
      "type": "Organization",
      "properties": {
        "description": "Official Police",
        "context": "The official police force that had abandoned certain mysteries as hopeless"
      }
    },
    {
      "id": "8",
      "type": "Organization",
      "properties": {
        "description": "Daily Press",
        "context": "The newspaper through which Watson occasionally heard of Holmes activities"
      }
    },
    {
      "id": "9",
      "type": "Location",
      "properties": {
        "description": "Baker Street",
        "context": "Holmes lodgings, also associated with Watson past wooing and dark incidents"
      }
    },
    {
      "id": "10",
      "type": "Location",
      "properties": {
        "description": "Odessa",
        "context": "City where Holmes was summoned for the Trepoff murder case"
      }
    },
    {
      "id": "11",
      "type": "Location",
      "properties": {
        "description": "Trincomalee",
        "context": "Location of the singular tragedy of the Atkinson Brothers"
      }
    },
    {
      "id": "12",
      "type": "Location",
      "properties": {
        "description": "Holland",
        "context": "Country whose reigning family Holmes undertook a mission for"
      }
    },
    {
      "id": "13",
      "type": "Event",
      "properties": {
        "description": "Trepoff Murder",
        "context": "A murder case in Odessa to which Holmes was summoned"
      }
    },
    {
      "id": "14",
      "type": "Event",
      "properties": {
        "description": "Tragedy of the Atkinson Brothers",
        "context": "A singular tragedy at Trincomalee cleared up by Holmes"
      }
    },
    {
      "id": "15",
      "type": "Event",
      "properties": {
        "description": "Mission for the Reigning Family of Holland",
        "context": "A delicate and successful mission accomplished by Holmes for the Dutch royal family"
      }
    },
    {
      "id": "16",
      "type": "Event",
      "properties": {
        "description": "Watson Marriage",
        "context": "Watson marriage which caused a drift between him and Holmes"
      }
    },
    {
      "id": "17",
      "type": "Work",
      "properties": {
        "description": "The Study in Scarlet",
        "context": "A referenced work associated with dark incidents and Baker Street"
      }
    },
    {
      "id": "18",
      "type": "Date",
      "properties": {
        "description": "Twentieth of March 1888",
        "context": "The specific night Watson returned through Baker Street and desired to see Holmes"
      }
    },
    {
      "id": "19",
      "type": "Substance",
      "properties": {
        "description": "Cocaine",
        "context": "A drug used by Holmes, alternating with periods of ambition and fierce energy"
      }
    }
  ],

  "relationships": [
    {
      "source": "3",
      "target": "1",
      "type": "COMPANION_OF",
      "properties": {
        "status": "estranged due to Watson marriage",
        "duration": "long-term"
      }
    },
    {
      "source": "3",
      "target": "1",
      "type": "NARRATES_ABOUT",
      "properties": {
        "context": "Watson is the first-person narrator describing Holmes and his activities"
      }
    },
    {
      "source": "3",
      "target": "1",
      "type": "VISITS",
      "properties": {
        "date": "Twentieth of March 1888",
        "location": "Baker Street"
      }
    },
    {
      "source": "1",
      "target": "2",
      "type": "ADMIRES",
      "properties": {
        "nature": "intellectual admiration, not romantic",
        "uniqueness": "only woman of significance to Holmes"
      }
    },
    {
      "source": "3",
      "target": "9",
      "type": "VISITS",
      "properties": {
        "date": "Twentieth of March 1888"
      }
    },
    {
      "source": "1",
      "target": "15",
      "type": "ACCOMPLISHED",
      "properties": {
        "outcome": "successful",
        "manner": "delicate"
      }
    },
    {
      "source": "1",
      "target": "19",
      "type": "USES",
      "properties": {
        "pattern": "alternating with periods of ambition and fierce energy"
      }
    },
    {
      "source": "1",
      "target": "7",
      "type": "SURPASSES",
      "properties": {
        "context": "Holmes solves cases the official police deemed hopeless"
      }
    },
    {
      "source": "1",
      "target": "8",
      "type": "MENTIONED_IN",
      "properties": {
        "context": "Holmes activities were reported in the daily press"
      }
    },
    {
      "source": "3",
      "target": "8",
      "type": "INFORMED_BY",
      "properties": {
        "context": "Watson learned of Holmes cases through the daily press"
      }
    }
  ]
}
"""

# human_prompt = HumanMessagePromptTemplate(
#     prompt=PromptTemplate(template=user_prompt)
# )

# chat_prompt = ChatPromptTemplate.from_messages([system_message, human_prompt])

In [ ]:
# llm_transformer = LLMGraphTransformer(
#     llm=llm,
#     allowed_nodes=["Person", "Location", "Organization", "Object", "Case"],
#     allowed_relationships=[
#         "KNOWS",
#         "LIVES_IN",
#         "VISITS",
#         "WORKS_FOR",
#         "OWNS",
#         "INVOLVED_IN",
#         "INVESTIGATES",
#         "ASSOCIATED_WITH"
#     ],
#     ignore_tool_usage=True,
#     strict_mode=True
# )

# llm.invoke(f"System prompt: {system_prompt}, user prompt: {user_prompt}, text: {documents[1]}")

# llm_transformer = LLMGraphTransformer(
#     llm=llm,
#     prompt=chat_prompt
# )

graph_documents = llm_transformer.convert_to_graph_documents(documents[:5])

In [65]:
graph_documents

[GraphDocument(nodes=[Node(id='Bohemia', type='Location', properties={}), Node(id='I', type='Case', properties={}), Node(id='Scandal', type='Case', properties={})], relationships=[], source=Document(metadata={'source': '../data/chapters/chapter_1.txt', 'id': 'c2f6eddc7d719fe10c8efa897a98d18c'}, page_content='I. A SCANDAL IN BOHEMIA\n\n\nI.')),
 GraphDocument(nodes=[Node(id='Irene Adler', type='Person', properties={}), Node(id='Sherlock Holmes', type='Person', properties={})], relationships=[Relationship(source=Node(id='Irene Adler', type='Person', properties={}), target=Node(id='Sherlock Holmes', type='Person', properties={}), type='ASSOCIATED_WITH', properties={})], source=Document(metadata={'source': '../data/chapters/chapter_1.txt', 'id': 'edc20a161184fb27059d4b2e88d4e9cd'}, page_content='To Sherlock Holmes she is always _the_ woman. I have seldom heard him\nmention her under any other name. In his eyes she eclipses and\npredominates the whole of her sex. It was not that he felt any

In [34]:
graph.add_graph_documents(
    graph_documents=graph_documents,
    baseEntityLabel=False,
    include_source=True
)

In [9]:
from langchain_neo4j.graphs.graph_document import GraphDocument, Node, Relationship


def serialize_entitiy(entity):
    return Node(id=entity["id"], type=entity["type"], properties=entity["properties"])

def serialize_relationship(relation, nodes: list[Node]):
    source = next(node for node in nodes if node.id == relation["source"])
    target = next(node for node in nodes if node.id == relation["target"])
    return Relationship(source=source, target=target, type=relation["type"], properties=relation["properties"])

In [10]:
result = {
  "nodes": [
    {
      "id": "1",
      "type": "Person",
      "properties": {
        "description": "Sherlock Holmes",
        "context": "A precise, cold and balanced reasoning machine, deeply attracted to the study of crime",
        "traits": ["logical", "emotionally detached", "bohemian", "observant"],
        "physical_description": "tall, spare figure",
        "habits": ["cocaine use", "pacing", "studying crime"]
      }
    },
    {
      "id": "2",
      "type": "Person",
      "properties": {
        "description": "Irene Adler",
        "context": "The only woman of significance to Holmes, described as late and of dubious memory",
        "status": "deceased"
      }
    },
    {
      "id": "3",
      "type": "Person",
      "properties": {
        "description": "Dr. Watson",
        "context": "Narrator, former companion of Holmes, medical doctor returned to civil practice",
        "status": "married",
        "occupation": "medical doctor"
      }
    },
    {
      "id": "4",
      "type": "Person",
      "properties": {
        "description": "Trepoff",
        "context": "Victim of a murder case in Odessa"
      }
    },
    {
      "id": "5",
      "type": "Person",
      "properties": {
        "description": "Atkinson Brothers",
        "context": "Involved in a singular tragedy at Trincomalee"
      }
    },
    {
      "id": "6",
      "type": "Organization",
      "properties": {
        "description": "Reigning Family of Holland",
        "context": "Royal family of Holland for whom Holmes accomplished a delicate mission"
      }
    },
    {
      "id": "7",
      "type": "Organization",
      "properties": {
        "description": "Official Police",
        "context": "The official police force that had abandoned certain mysteries as hopeless"
      }
    },
    {
      "id": "8",
      "type": "Organization",
      "properties": {
        "description": "Daily Press",
        "context": "The newspaper through which Watson occasionally heard of Holmes activities"
      }
    },
    {
      "id": "9",
      "type": "Location",
      "properties": {
        "description": "Baker Street",
        "context": "Holmes lodgings, also associated with Watson past wooing and dark incidents"
      }
    },
    {
      "id": "10",
      "type": "Location",
      "properties": {
        "description": "Odessa",
        "context": "City where Holmes was summoned for the Trepoff murder case"
      }
    },
    {
      "id": "11",
      "type": "Location",
      "properties": {
        "description": "Trincomalee",
        "context": "Location of the singular tragedy of the Atkinson Brothers"
      }
    },
    {
      "id": "12",
      "type": "Location",
      "properties": {
        "description": "Holland",
        "context": "Country whose reigning family Holmes undertook a mission for"
      }
    },
    {
      "id": "13",
      "type": "Event",
      "properties": {
        "description": "Trepoff Murder",
        "context": "A murder case in Odessa to which Holmes was summoned"
      }
    },
    {
      "id": "14",
      "type": "Event",
      "properties": {
        "description": "Tragedy of the Atkinson Brothers",
        "context": "A singular tragedy at Trincomalee cleared up by Holmes"
      }
    },
    {
      "id": "15",
      "type": "Event",
      "properties": {
        "description": "Mission for the Reigning Family of Holland",
        "context": "A delicate and successful mission accomplished by Holmes for the Dutch royal family"
      }
    },
    {
      "id": "16",
      "type": "Event",
      "properties": {
        "description": "Watson Marriage",
        "context": "Watson marriage which caused a drift between him and Holmes"
      }
    },
    {
      "id": "17",
      "type": "Work",
      "properties": {
        "description": "The Study in Scarlet",
        "context": "A referenced work associated with dark incidents and Baker Street"
      }
    },
    {
      "id": "18",
      "type": "Date",
      "properties": {
        "description": "Twentieth of March 1888",
        "context": "The specific night Watson returned through Baker Street and desired to see Holmes"
      }
    },
    {
      "id": "19",
      "type": "Substance",
      "properties": {
        "description": "Cocaine",
        "context": "A drug used by Holmes, alternating with periods of ambition and fierce energy"
      }
    }
  ],

  "relationships": [
    {
      "source": "3",
      "target": "1",
      "type": "COMPANION_OF",
      "properties": {
        "status": "estranged due to Watson marriage",
        "duration": "long-term"
      }
    },
    {
      "source": "3",
      "target": "1",
      "type": "NARRATES_ABOUT",
      "properties": {
        "context": "Watson is the first-person narrator describing Holmes and his activities"
      }
    },
    {
      "source": "3",
      "target": "1",
      "type": "VISITS",
      "properties": {
        "date": "Twentieth of March 1888",
        "location": "Baker Street"
      }
    },
    {
      "source": "1",
      "target": "2",
      "type": "ADMIRES",
      "properties": {
        "nature": "intellectual admiration, not romantic",
        "uniqueness": "only woman of significance to Holmes"
      }
    },
    {
      "source": "1",
      "target": "9",
      "type": "RESIDES_AT",
      "properties": {}
    },
    {
      "source": "3",
      "target": "9",
      "type": "FORMERLY_RESIDED_AT",
      "properties": {}
    },
    {
      "source": "3",
      "target": "9",
      "type": "VISITS",
      "properties": {
        "date": "Twentieth of March 1888"
      }
    },
    {
      "source": "1",
      "target": "13",
      "type": "INVESTIGATED",
      "properties": {}
    },
    {
      "source": "4",
      "target": "13",
      "type": "VICTIM_OF",
      "properties": {}
    },
    {
      "source": "13",
      "target": "10",
      "type": "OCCURRED_IN",
      "properties": {}
    },
    {
      "source": "1",
      "target": "14",
      "type": "INVESTIGATED",
      "properties": {}
    },
    {
      "source": "5",
      "target": "14",
      "type": "INVOLVED_IN",
      "properties": {}
    },
    {
      "source": "14",
      "target": "11",
      "type": "OCCURRED_IN",
      "properties": {}
    },
    {
      "source": "1",
      "target": "15",
      "type": "ACCOMPLISHED",
      "properties": {
        "outcome": "successful",
        "manner": "delicate"
      }
    },
    {
      "source": "15",
      "target": "6",
      "type": "COMMISSIONED_BY",
      "properties": {}
    },
    {
      "source": "6",
      "target": "12",
      "type": "LOCATED_IN",
      "properties": {}
    },
    {
      "source": "1",
      "target": "19",
      "type": "USES",
      "properties": {
        "pattern": "alternating with periods of ambition and fierce energy"
      }
    },
    {
      "source": "1",
      "target": "7",
      "type": "SURPASSES",
      "properties": {
        "context": "Holmes solves cases the official police deemed hopeless"
      }
    },
    {
      "source": "3",
      "target": "16",
      "type": "PARTICIPATED_IN",
      "properties": {}
    },
    {
      "source": "16",
      "target": "1",
      "type": "CAUSED_ESTRANGEMENT_FROM",
      "properties": {}
    },
    {
      "source": "3",
      "target": "17",
      "type": "REFERENCES",
      "properties": {}
    },
    {
      "source": "17",
      "target": "9",
      "type": "ASSOCIATED_WITH",
      "properties": {}
    },
    {
      "source": "3",
      "target": "18",
      "type": "EXPERIENCED",
      "properties": {}
    },
    {
      "source": "1",
      "target": "8",
      "type": "MENTIONED_IN",
      "properties": {
        "context": "Holmes activities were reported in the daily press"
      }
    },
    {
      "source": "3",
      "target": "8",
      "type": "INFORMED_BY",
      "properties": {
        "context": "Watson learned of Holmes cases through the daily press"
      }
    }
  ]
}

In [11]:
node_objects: list[Node] = []
for node in result["nodes"]:
    node_objects.append(serialize_entitiy(node))

relationship_objects: list[Relationship] = []
for rel in result["relationships"]:
    relationship_objects.append(serialize_relationship(rel, node_objects))

In [12]:
graph.add_graph_documents(
    [GraphDocument(nodes=node_objects, relationships=relationship_objects)],
    baseEntityLabel=False,
    include_source=False
)

In [22]:
embeddings = OllamaEmbeddings(
    model="mxbai-embed-large",
)

vector_index = Neo4jVector.from_existing_graph(
    embeddings,
    search_type="hybrid",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding"
)
vector_retriever = vector_index.as_retriever()

In [ ]:
driver = GraphDatabase.driver(
        uri = os.environ["NEO4J_URI"],
        auth = (os.environ["NEO4J_USERNAME"],
                os.environ["NEO4J_PASSWORD"]))

def create_fulltext_index(tx):
    query = '''
    CREATE FULLTEXT INDEX `fulltext_entity_id` 
    FOR (n:__Entity__) 
    ON EACH [n.id];
    '''
    tx.run(query)

# Function to execute the query
def create_index():
    with driver.session() as session:
        session.execute_write(create_fulltext_index)
        print("Fulltext index created successfully.")

# Call the function to create the index
try:
    create_index()
except:
    pass

# Close the driver connection
driver.close()

Fulltext index created successfully.


In [24]:

class Entities(BaseModel):
    """Identifying information about entities."""

    names: list[str] = Field(
        ...,
        description="All the person, organization, or business entities that "
        "appear in the text",
    )

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are extracting organization and person entities from the text.",
        ),
        (
            "human",
            "Use the given format to extract information from the following "
            "input: {question}",
        ),
    ]
)


entity_chain = llm.with_structured_output(Entities)

In [26]:
entity_chain.invoke("Who are Sherlock Holmes and Dr. Watson?")

Entities(names=['Sherlock Holmes', 'Dr. John Watson'])

In [27]:
def generate_full_text_query(input: str) -> str:
    words = [el for el in remove_lucene_chars(input).split() if el]
    if not words:
        return ""
    full_text_query = " AND ".join([f"{word}~2" for word in words])
    print(f"Generated Query: {full_text_query}")
    return full_text_query.strip()


# Fulltext index query
def graph_retriever(question: str) -> str:
    """
    Collects the neighborhood of entities mentioned
    in the question
    """
    result = ""
    entities = entity_chain.invoke(question)
    for entity in entities.names:
        response = graph.query(
            """CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit:2})
            YIELD node,score
            CALL {
              WITH node
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output
              UNION ALL
              WITH node
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS output
            }
            RETURN output LIMIT 50
            """,
            {"query": entity},
        )
        result += "\n".join([el['output'] for el in response])
    return result

In [28]:
print(graph_retriever("Who is Sherlock Holmes?"))

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=3, column=13, offset=116>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 116, 'line': 3, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit:2})\n            YIELD node,score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n 

Dr. Watson - COMPANION_OF -> Sherlock Holmes


In [29]:
def full_retriever(question: str):
    graph_data = graph_retriever(question)
    vector_data = [el.page_content for el in vector_retriever.invoke(question)]
    final_data = f"""Graph data:
{graph_data}
vector data:
{"#Document ". join(vector_data)}
    """
    return final_data

In [30]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
Use natural language and be concise.
Answer:"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
        {
            "context": full_retriever,
            "question": RunnablePassthrough(),
        }
    | prompt
    | llm
    | StrOutputParser()
)

In [31]:
chain.invoke(input="Who is Irene Adler? How was she related to Sherlock Holmes?")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (node) { ... }', position=<SummaryInputPosition line=3, column=13, offset=116>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 116, 'line': 3, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('fulltext_entity_id', $query, {limit:2})\n            YIELD node,score\n            CALL {\n              WITH node\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS output\n              UNION ALL\n              WITH node\n 

'{ "Irene Adler":"The only woman to outwit Sherlock Holmes. She was a opera singer who had a brief encounter with him." }'